# 05 Seasonal Box-Jenkins Models

By the end of this notebook, you should be able to:

- recognize seasonality that ordinary differencing does not remove;
- compute seasonal and combined regular-seasonal differences;
- interpret SARIMA notation $(p,d,q)(P,D,Q)_s$;
- fit a seasonal ARIMA model using `statsmodels`.

In [1]:
from lite_setup import ensure_packages
await ensure_packages()

Running outside JupyterLite; assuming packages are already installed.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from checks import check, check_between, check_close, check_columns
from boxjenkins_utils import (
    first_difference, second_difference, seasonal_difference, regular_then_seasonal_difference,
    acf_pacf_table, mean_zero_t_test, fit_arima, fit_sarima, parameter_table,
    forecast_table, ljung_box_table, arima_grid_search, plot_series,
    plot_acf_pacf_pair, plot_forecast,
)
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.precision', 4)
DATA_DIR = Path('data')
hotel = pd.read_csv(DATA_DIR / 'hotel_rooms.csv')
y = hotel['Rooms'].astype(float)
hotel.head()

,Month,Rooms
0,1,501
1,2,488
2,3,504
3,4,578
4,5,545


## Seasonal Differencing

For monthly data with annual seasonality, the seasonal period is $s=12$. A seasonal difference is

$$z_t = y_t-y_{t-12}.$$

If the series also has local trend, use both a regular and seasonal difference:

$$z_t = (1-B)(1-B^{12})y_t.$$

In SARIMA notation, $(p,d,q)(P,D,Q)_s$, `d` is the number of regular differences and `D` is the number of seasonal differences.

In [3]:
y_root = y ** 0.25
regular = first_difference(y_root)
seasonal = seasonal_difference(y_root, 12)
combined = regular_then_seasonal_difference(y_root, 12)
fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=False)
axes[0].plot(hotel['Month'], y, marker='o')
axes[0].set_title('Monthly hotel rooms')
axes[0].set_ylabel('Rooms')
axes[1].plot(np.arange(1, len(y_root) + 1), y_root, marker='o', color='tab:purple')
axes[1].set_title('Fourth-root transformed series')
axes[2].plot(np.arange(13, len(y_root) + 1), seasonal, marker='o', color='tab:orange')
axes[2].set_title('Seasonal difference of transformed series')
axes[3].plot(np.arange(14, len(y_root) + 1), combined, marker='o', color='tab:green')
axes[3].set_title('Regular plus seasonal difference')
axes[3].set_xlabel('Month')
plt.tight_layout()

## Seasonal AR and MA Terms

A seasonal MA(1) term at period 12 means the current value depends on the shock 12 periods ago:

$$z_t = a_t - \Theta_1 a_{t-12}.$$

A seasonal AR(1) term at period 12 means the current value depends on the working-series value 12 periods ago:

$$z_t = \Phi_1 z_{t-12} + a_t.$$

In `statsmodels`, these appear in `seasonal_order=(P, D, Q, s)`.

In [4]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
plot_acf(seasonal, lags=36, ax=axes[0, 0], title='ACF: seasonal difference')
plot_pacf(seasonal, lags=36, ax=axes[0, 1], title='PACF: seasonal difference', method='ywmle')
plot_acf(combined, lags=36, ax=axes[1, 0], title='ACF: regular + seasonal difference')
plot_pacf(combined, lags=36, ax=axes[1, 1], title='PACF: regular + seasonal difference', method='ywmle')
plt.tight_layout()

## A Historical-Software SARIMA Fit

The old R notes fit a seasonal model to the fourth-root transformed hotel data with seasonal differencing and a seasonal MA term. The following `statsmodels` version uses

$$\text{SARIMA}(5,0,0)(0,1,1)_{12}$$

on $y_t^{1/4}$. This mirrors the historical workflow and gives a useful template for seasonal ARIMA syntax.

In [5]:
sarima = fit_sarima(y_root, order=(5, 0, 0), seasonal_order=(0, 1, 1, 12), trend='n')
parameter_table(sarima).round(4)

,estimate,std_error,z_or_t,p_value
ar.L1,0.5396,0.0807,6.6834,0.0000
ar.L2,0.2903,0.1008,2.8795,0.0040
ar.L3,-0.1630,0.1110,-1.4690,0.1418
ar.L4,0.1881,0.0997,1.8863,0.0592
ar.L5,0.1239,0.0783,1.5829,0.1135
ma.S.L12,-0.6206,0.0811,-7.6549,0.0000
sigma2,0.0008,0.0001,8.2917,0.0000


In [6]:
resid = pd.Series(sarima.resid).dropna().reset_index(drop=True)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_acf(resid, lags=36, ax=axes[0], title='Residual ACF')
plot_pacf(resid, lags=36, ax=axes[1], title='Residual PACF', method='ywmle')
plt.tight_layout()

In [7]:
ljung_box_table(resid, lags=(12, 24, 36)).round(4)

,lb_stat,lb_pvalue
12,185.2091,0.0
24,191.3083,0.0
36,191.3205,0.0


## Forecasting Back On The Original Scale

Because the model was fitted to $y_t^{1/4}$, forecasts must be transformed back by raising to the fourth power. A simple back-transform of the point forecast is easy; rigorous intervals on the original scale require more care because nonlinear transformations change interval symmetry.

In [8]:
fc_root = forecast_table(sarima, steps=24)
fc_rooms = fc_root.copy()
for col in ['forecast', 'lower', 'upper']:
    fc_rooms[col] = np.maximum(fc_rooms[col], 0) ** 4
fc_rooms.head(12).round(1)

,forecast,lower,upper
step,,,
1,836.5,802.4,871.7
2,772.3,735.9,810.0
3,779.2,738.1,821.9
4,877.8,831.6,925.9
5,867.0,818.9,917.3
6,987.3,931.4,1045.6
7,1151.7,1085.5,1221.0
8,1176.3,1106.1,1249.7
9,904.2,844.7,966.9


In [9]:
forecast_x = np.arange(len(y) + 1, len(y) + 25)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(hotel['Month'], y, label='Observed')
ax.plot(forecast_x, fc_rooms['forecast'], marker='o', label='Forecast')
ax.fill_between(forecast_x, fc_rooms['lower'].to_numpy(), fc_rooms['upper'].to_numpy(), alpha=0.2, label='Approx. interval')
ax.set_title('Hotel room forecasts from SARIMA on fourth-root scale')
ax.set_xlabel('Month')
ax.set_ylabel('Rooms')
ax.legend()
plt.tight_layout()

## Common Seasonal Variants From The Software Notes

The historical software notes also show an airline-style seasonal model after a log transform:

$$\text{SARIMA}(0,1,1)(0,1,1)_{12}.$$

This model uses one regular difference, one seasonal difference, one nonseasonal MA term, and one seasonal MA term. It is common for monthly series with trend and multiplicative seasonal variation. The exact data set in the old notes is different, but the package syntax is the same.

In [10]:
log_rooms = np.log(y)
airline_style = fit_sarima(log_rooms, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12), trend='n')
parameter_table(airline_style).round(4)

,estimate,std_error,z_or_t,p_value
ma.L1,-0.9476,0.0319,-29.7096,0.0
ma.S.L12,-0.5549,0.0829,-6.6894,0.0
sigma2,0.0004,0.0001,8.2730,0.0


## Regression With ARIMA Errors

The software notes also connect Box-Jenkins ideas to time-series regression. If deterministic predictors such as trend, month indicators, or interventions explain part of the mean but the residuals are autocorrelated, the model can be written as a regression with ARIMA errors.

In Python, this is usually handled with `SARIMAX(..., exog=X)`: the `exog` matrix contains regression predictors, while `order=` and `seasonal_order=` describe the ARIMA structure of the error term. That topic connects this module back to the earlier time-series regression material. The diagnostic idea is the same: after fitting the mean model, check whether remaining residual autocorrelation still needs an ARIMA error model.

## Practice Prompt

A monthly series has strong ACF spikes at lags 12, 24, and 36 even after a regular first difference. What is the next transformation you should consider?

Reference idea: use a seasonal difference with lag 12, and if both trend and seasonality remain, consider the combined $(1-B)(1-B^{12})$ transformation.